# Book Market Analytics: Books to Scrape
**Data Science 500 | End-to-end pipeline: scraping, cleaning, visualisation, linear regression**

**Workflow:** 1) Scrape all 1,000 books with BeautifulSoup, 2) clean the raw data, 3) explore it visually, 4) model price with Linear Regression, 5) save results for the report.

**How to run:** `pip install -r requirements.txt`, then *Kernel > Restart & Run All*. The first run scrapes the site (about 3-5 minutes, politely rate-limited) and saves `data/books_raw.csv`. Later runs reuse that file unless `FORCE_SCRAPE = True`.

## 0. Setup and configuration

In [ ]:
import os, re, json, time, warnings
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split, KFold

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

BASE_URL = "https://books.toscrape.com/catalogue/"
MAX_PAGES = 50            # the catalogue has 50 pages of 20 books = 1,000 books
REQUEST_DELAY = 0.15      # seconds between requests (be polite to the server)
FORCE_SCRAPE = False      # True = ignore the saved raw file and scrape again

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
RAW_CSV, CLEAN_CSV = "data/books_raw.csv", "data/books_clean.csv"

## 1. Data acquisition (web scraping with BeautifulSoup)
Each catalogue page lists 20 books (title, price, star rating, availability). Each book's detail page adds the **genre** (from the breadcrumb), the **description**, the exact **stock quantity**, UPC and review count. The scraper visits all 50 listing pages and every book's detail page.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (student data-science project)"})


def get_soup(url, retries=3):
    """Download a page and return a BeautifulSoup object (retries on failure)."""
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()
            resp.encoding = "utf-8"          # avoids the 'Â£' encoding glitch
            return BeautifulSoup(resp.text, "html.parser")
        except requests.RequestException as exc:
            print(f"  attempt {attempt} failed for {url}: {exc}")
            time.sleep(1.5 * attempt)
    return None


def parse_listing_page(soup, page_url):
    """Extract title, raw price, star rating word, availability and detail URL for each book."""
    books = []
    for card in soup.select("article.product_pod"):
        link = card.h3.a
        rating_classes = card.select_one("p.star-rating")["class"]       # e.g. ['star-rating', 'Three']
        books.append({
            "title": link.get("title", link.get_text(strip=True)),
            "price_raw": card.select_one("p.price_color").get_text(strip=True),
            "rating_word": rating_classes[1] if len(rating_classes) > 1 else None,
            "availability_listing": card.select_one("p.availability").get_text(strip=True),
            "url": urljoin(page_url, link["href"]),
        })
    return books


def parse_detail_page(soup):
    """Extract genre, description and product-information table fields."""
    crumbs = [li.get_text(strip=True) for li in soup.select("ul.breadcrumb li")]
    genre = crumbs[2] if len(crumbs) >= 3 else None
    desc_header = soup.select_one("#product_description")
    description = None
    if desc_header is not None and desc_header.find_next_sibling("p") is not None:
        description = desc_header.find_next_sibling("p").get_text(strip=True)
    info = {row.th.get_text(strip=True): row.td.get_text(strip=True)
            for row in soup.select("table.table-striped tr")}
    return {
        "genre": genre,
        "description": description,
        "upc": info.get("UPC"),
        "product_type": info.get("Product Type"),
        "availability_detail": info.get("Availability"),
        "num_reviews": info.get("Number of reviews"),
    }


def scrape_catalogue(max_pages=MAX_PAGES):
    """Scrape every listing page and every detail page; return a raw DataFrame."""
    records = []
    for page in range(1, max_pages + 1):
        page_url = f"{BASE_URL}page-{page}.html"
        soup = get_soup(page_url)
        if soup is None:
            print(f"Skipping page {page} (could not download)")
            continue
        for book in parse_listing_page(soup, page_url):
            time.sleep(REQUEST_DELAY)
            detail = get_soup(book["url"])
            if detail is not None:
                book.update(parse_detail_page(detail))
            records.append(book)
        print(f"Page {page}/{max_pages} done - {len(records)} books so far")
    return pd.DataFrame(records)


if os.path.exists(RAW_CSV) and not FORCE_SCRAPE:
    raw = pd.read_csv(RAW_CSV)
    print(f"Loaded saved raw data: {raw.shape[0]} rows")
else:
    raw = scrape_catalogue()
    raw.to_csv(RAW_CSV, index=False)
    print(f"Scraped and saved {raw.shape[0]} rows to {RAW_CSV}")
raw.head()

## 2. Data cleaning and preparation

In [ ]:
print("Shape:", raw.shape)
print("\nMissing values per column:\n", raw.isna().sum())
print("\nDuplicate rows (by UPC):", raw.duplicated(subset="upc").sum())
print("\nSample raw price / availability values:")
print(raw[["price_raw", "availability_listing", "availability_detail"]].head(3))

In [ ]:
df = raw.copy()

# 1) Remove exact duplicates (same UPC = same product).
before = len(df)
df = df.drop_duplicates(subset="upc").reset_index(drop=True)
print(f"Duplicates removed: {before - len(df)}")

# 2) Price: strip the currency symbol and any stray encoding characters, convert to float.
df["price"] = pd.to_numeric(df["price_raw"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")

# 3) Rating: map the word to a number 1-5 (the model needs a numeric feature).
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating_word"].map(rating_map)

# 4) Stock: In stock / Out of stock flag + quantity from "In stock (22 available)".
df["in_stock"] = df["availability_listing"].astype(str).str.contains("In stock", case=False).astype(int)
qty = df["availability_detail"].astype(str).str.extract(r"\((\d+)\s+available\)")[0]
df["stock_qty"] = pd.to_numeric(qty, errors="coerce").fillna(0).astype(int)
# Justification: if the quantity is not shown, an in-stock book still has at least 1 copy.
df.loc[(df["in_stock"] == 1) & (df["stock_qty"] == 0), "stock_qty"] = 1

# 5) Reviews: numeric.
df["num_reviews"] = pd.to_numeric(df["num_reviews"], errors="coerce").fillna(0).astype(int)

# 6) Text fields: missing descriptions -> empty string + flag; clean text; derive length features.
df["has_description"] = df["description"].notna().astype(int)
df["description"] = df["description"].fillna("").astype(str)
df["description_clean"] = (df["description"].str.lower()
                           .str.replace(r"[^a-z\s]", " ", regex=True)
                           .str.replace(r"\s+", " ", regex=True).str.strip())
df["desc_word_count"] = df["description_clean"].str.split().str.len()
df["title"] = df["title"].astype(str).str.strip()
df["title_length"] = df["title"].str.len()
df["genre"] = df["genre"].fillna("Unknown").astype(str).str.strip()

# 7) Drop rows missing the essentials (price / rating); report how many.
essential = ["price", "rating"]
n_bad = df[essential].isna().any(axis=1).sum()
df = df.dropna(subset=essential).reset_index(drop=True)
print(f"Rows dropped for missing price/rating: {n_bad}")

# 8) Outlier check (IQR). Outliers are kept: they are genuine prices, not errors.
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[(df["price"] < q1 - 1.5 * iqr) | (df["price"] > q3 + 1.5 * iqr)]
print(f"Price outliers by IQR rule: {len(outliers)} (kept; real values)")

clean = df[["title", "genre", "price", "rating", "rating_word", "in_stock", "stock_qty", "num_reviews",
            "has_description", "desc_word_count", "title_length", "description_clean", "upc", "url"]]
clean.to_csv(CLEAN_CSV, index=False)
print(f"\nClean dataset saved: {clean.shape[0]} rows x {clean.shape[1]} columns -> {CLEAN_CSV}")
clean.info()

In [ ]:
clean.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
# Simple text preparation check: most common meaningful words in descriptions.
stop = set("the a an and of to in is it that for on with as by his her their he she they this are was be from at or "
           "who has have but not you your will its one into who him them when what which more than also been "
           "can all out about up so if only".split())
words = pd.Series(" ".join(clean["description_clean"]).split())
words = words[~words.isin(stop) & (words.str.len() > 3)]
print(words.value_counts().head(15))

## 3. Data visualisation and exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(clean["price"], bins=30, kde=True, color="#0056b3", ax=axes[0])
axes[0].set(title="Distribution of book prices", xlabel="Price (GBP)", ylabel="Number of books")
axes[0].axvline(clean["price"].mean(), color="red", ls="--", label=f"Mean £{clean['price'].mean():.2f}")
axes[0].legend()
sns.countplot(x="rating", data=clean, color="#12336e", ax=axes[1])
axes[1].set(title="Number of books per star rating", xlabel="Star rating", ylabel="Number of books")
plt.tight_layout(); plt.savefig("figures/01_price_and_rating_distribution.png", dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="rating", y="price", data=clean, color="#9ec5f0")
sns.pointplot(x="rating", y="price", data=clean, estimator=np.mean, color="red", errorbar=None, markers="D")
plt.title("Price by star rating (red = mean)"); plt.xlabel("Star rating"); plt.ylabel("Price (GBP)")
plt.tight_layout(); plt.savefig("figures/02_price_by_rating.png", dpi=150); plt.show()
print(clean.groupby("rating")["price"].agg(["count", "mean", "median"]).round(2))

In [ ]:
genre_stats = (clean.groupby("genre")["price"].agg(["count", "mean", "median"]).round(2)
               .sort_values("count", ascending=False))
top = genre_stats.head(15).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(y="genre", x="count", data=top, color="#12336e", ax=axes[0])
axes[0].set(title="15 largest genres by number of books", xlabel="Books", ylabel="")
sns.barplot(y="genre", x="mean", data=top.sort_values("mean", ascending=False), color="#0056b3", ax=axes[1])
axes[1].set(title="Average price in those genres", xlabel="Mean price (GBP)", ylabel="")
plt.tight_layout(); plt.savefig("figures/03_genre_counts_and_prices.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.scatterplot(x="desc_word_count", y="price", data=clean, alpha=0.4, ax=axes[0])
axes[0].set(title="Price vs description length", xlabel="Words in description", ylabel="Price (GBP)")
sns.scatterplot(x="stock_qty", y="price", data=clean, alpha=0.4, ax=axes[1])
axes[1].set(title="Price vs stock quantity", xlabel="Copies in stock", ylabel="Price (GBP)")
sns.scatterplot(x="title_length", y="price", data=clean, alpha=0.4, ax=axes[2])
axes[2].set(title="Price vs title length", xlabel="Characters in title", ylabel="Price (GBP)")
plt.tight_layout(); plt.savefig("figures/04_price_vs_numeric_features.png", dpi=150); plt.show()

In [ ]:
num_cols = ["price", "rating", "stock_qty", "num_reviews", "desc_word_count", "title_length"]
corr = clean[num_cols].corr()
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation between numeric variables")
plt.tight_layout(); plt.savefig("figures/05_correlation_heatmap.png", dpi=150); plt.show()
print("Correlation of each variable with price:\n", corr["price"].drop("price").round(3))

In [ ]:
common = genre_stats[genre_stats["count"] >= 15].index
pivot = (clean[clean["genre"].isin(common)].pivot_table(index="genre", columns="rating", values="price", aggfunc="mean"))
plt.figure(figsize=(9, 8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Mean price (GBP)"})
plt.title("Mean price by genre and star rating (genres with 15+ books)"); plt.xlabel("Star rating"); plt.ylabel("")
plt.tight_layout(); plt.savefig("figures/06_genre_rating_price_heatmap.png", dpi=150); plt.show()

### Interpretation of visual findings
Write 4-6 sentences here **using the numbers printed above** (price spread and mean, whether higher star ratings cost more, which genres are priciest, and how weak or strong the correlations with price are). Typical points to check: Is the price distribution roughly flat or skewed? Do the box plots for each rating overlap heavily? Are the correlations with price close to zero? Does the genre ranking change when you only look at genres with enough books?

## 4. Predictive modelling: Linear Regression
**Target:** `price` (GBP). **Features:** star rating, stock quantity, number of reviews, description word count, title length, has-description flag, and the book's genre (one-hot encoded; genres with fewer than 10 books are grouped as *Other* so the model does not overfit tiny categories).

All features are available publicly on each product page, so the model estimates price from catalogue information only.

In [ ]:
model_df = clean.copy()
genre_counts = model_df["genre"].value_counts()
rare = genre_counts[genre_counts < 10].index
model_df["genre_grouped"] = model_df["genre"].where(~model_df["genre"].isin(rare), "Other")

numeric_features = ["rating", "stock_qty", "num_reviews", "desc_word_count", "title_length", "has_description"]
X = pd.concat([model_df[numeric_features],
               pd.get_dummies(model_df["genre_grouped"], prefix="genre", drop_first=True, dtype=int)], axis=1)
# num_reviews is constant on this site; drop zero-variance columns (they carry no information).
X = X.loc[:, X.nunique() > 1]
y = model_df["price"]
print("Feature matrix:", X.shape, "| grouped rare genres:", len(rare))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
model = LinearRegression().fit(X_train, y_train)

def evaluate(name, m, Xa, ya):
    pred = m.predict(Xa)
    return {"model": name, "MAE": mean_absolute_error(ya, pred),
            "RMSE": np.sqrt(mean_squared_error(ya, pred)), "R2": r2_score(ya, pred)}

metrics = pd.DataFrame([evaluate("Baseline (mean) - test", baseline, X_test, y_test),
                        evaluate("Linear Regression - train", model, X_train, y_train),
                        evaluate("Linear Regression - test", model, X_test, y_test)]).round(3)
cv = cross_val_score(LinearRegression(), X, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="r2")
print(metrics.to_string(index=False))
print(f"\n5-fold cross-validated R2: mean {cv.mean():.3f} (std {cv.std():.3f})")

In [ ]:
coefs = pd.Series(model.coef_, index=X.columns).sort_values()
print(f"Intercept: {model.intercept_:.2f}")
display_n = pd.concat([coefs.head(8), coefs.tail(8)])
plt.figure(figsize=(8, 6))
display_n.plot(kind="barh", color=np.where(display_n < 0, "#c0392b", "#0056b3"))
plt.title("Largest linear regression coefficients (effect on price, GBP)"); plt.xlabel("Coefficient")
plt.tight_layout(); plt.savefig("figures/07_model_coefficients.png", dpi=150); plt.show()

pred = model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, pred, alpha=0.5)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--"); axes[0].set(title="Actual vs predicted price", xlabel="Actual (GBP)", ylabel="Predicted (GBP)")
axes[1].scatter(pred, y_test - pred, alpha=0.5); axes[1].axhline(0, color="red", ls="--")
axes[1].set(title="Residuals vs predicted", xlabel="Predicted (GBP)", ylabel="Residual (GBP)")
plt.tight_layout(); plt.savefig("figures/08_actual_vs_predicted_and_residuals.png", dpi=150); plt.show()

In [ ]:
summary = {
    "n_books": int(len(clean)), "n_genres": int(clean["genre"].nunique()),
    "price_mean": round(float(clean["price"].mean()), 2), "price_median": round(float(clean["price"].median()), 2),
    "price_min": float(clean["price"].min()), "price_max": float(clean["price"].max()),
    "mean_rating": round(float(clean["rating"].mean()), 2),
    "in_stock_share": round(float(clean["in_stock"].mean()), 3),
    "corr_with_price": clean[num_cols].corr()["price"].drop("price").dropna().round(3).to_dict(),
    "test_MAE": round(float(metrics.iloc[2]["MAE"]), 3), "test_RMSE": round(float(metrics.iloc[2]["RMSE"]), 3),
    "test_R2": round(float(metrics.iloc[2]["R2"]), 3), "baseline_MAE": round(float(metrics.iloc[0]["MAE"]), 3),
    "cv_R2_mean": round(float(cv.mean()), 3), "cv_R2_std": round(float(cv.std()), 3),
    "n_features": int(X.shape[1]),
}
with open("results/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
coefs.rename("coefficient").to_csv("results/model_coefficients.csv")
metrics.to_csv("results/model_metrics.csv", index=False)
print(json.dumps(summary, indent=2))

### Interpretation and limitations
Use `results/summary.json` and the metrics above to answer:
- **How good is the model?** Compare test MAE and R² with the mean-price baseline. An R² near 0 means the features explain very little of the price.
- **Which features matter?** Look at the coefficients (and whether the sign makes business sense).
- **Limitations to discuss:** Books to Scrape is a practice sandbox, so its prices are not real market prices and may be largely random; only a few catalogue features are available (no author, publisher, page count, format); linear regression assumes straight-line effects; many genres have few books; residual plots show whether errors are even across price levels.

## 5. Conclusions
Summarise in 3-5 bullets: the key catalogue insights, the model result (with MAE and R²), the practical meaning, and recommended next steps (for example, adding author, publisher or page count and trying non-linear models). Then transfer these into `REPORT_TEMPLATE.md`.